<b>Transform Payments Data

1. Extract date and time from payment_timestamp and create new columns payment_date and payment_time
2. Map payment_status to descriptive values (1‑Success, 2‑Pending, 3‑Cancelled, 4‑Failed)
3. Write the transformed data to the Silver schema

In [0]:
df = spark.table('gizmobox.bronze.payments')
display(df)

<b> 1.Extract Date and Time from payment_timestemp
<br>
> [Documention for date format function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/date_format)

In [0]:
from pyspark.sql import functions as f

df_extracted_payments = (
                            df
                            .select('payment_id', 'customer_id', f.date_format('payment_date', 'yyyy-MM-dd').cast('date').alias('payment_date'),
                                      f.date_format('payment_date', 'HH:mm:ss').alias('payment_time'), 'payment_status', 'payment_method'
                            )
)
display(df_extracted_payments)

<b>2. Map payment_status to contain descriptive valuesn
<br>
>(1-Success, 2-Peding, 3-Cancelled, 4-Failed)

In [0]:
df = spark.table("gizmobox.bronze.payments")

df_mapped_payments = (
    df.select(
        "payment_id",
        "customer_id",
        f.to_date("payment_date").alias("payment_date"),
        f.date_format("payment_date", "HH:mm:ss").alias("payment_time"),
        f.when(f.col("payment_status") == 1, "SUCCESS")
         .when(f.col("payment_status") == 2, "PEDING")
         .when(f.col("payment_status") == 3, "CANCELLED")
         .when(f.col("payment_status") == 4, "FAILED")
         .otherwise("UNKNOWN")
         .alias("payment_status"),
        "payment_method"
    )
)
display(df_mapped_payments)

<b> 3. Write transformed data to the Silver schema 

In [0]:
df_mapped_payments.writeTo('gizmobox.silver.py_payments').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_payments